# LLM 생성 테스트는 코드 변화를 얼마나 버티는가

진행 중인 실험의 일부를 줄여서 직접 실행해 보는 강의 자료입니다.

- 대상: json-schema-validator (Java, JSON Schema 검증 라이브러리)
- 비교 집단: 개발자가 작성한 테스트(Dev), Ouédraogo 방식으로 생성한 테스트, KTester 방식으로 생성한 테스트
- 진행: 시점 t에서 테스트를 한 번 생성하고, 이후 commit 세 개에서 수정 없이 실행해 Dev 테스트와 비교합니다.

**Google Colab** 기준입니다. 위에서부터 순서대로 실행하세요. 셀이 쓰는 함수는 강의 저장소의 `tools/lecture.py`에 있습니다.

## 1. 환경 준비

| 도구 | 하는 일 |
|---|---|
| JDK 17, Maven | Java 컴파일과 실행, 대상 프로젝트 빌드 |
| JUnit 콘솔 런처 | 테스트를 메서드 단위로 실행하고 결과 수집 |
| Mockito | 생성된 테스트가 쓰는 mock 라이브러리 |
| JaCoCo | 테스트가 production 코드의 어느 줄을 실행했는지 측정 |

### 1-1. 강의 저장소 받기, JDK와 Maven 설치 (1분 안팎)

In [ ]:
import os, sys, subprocess, pathlib, json
BRANCH = "retarget-classes"
LECTURE = pathlib.Path(os.environ.get("LECTURE_REPO", "/content/test-survival-lecture"))
if not LECTURE.exists():
    subprocess.run(["git", "clone", "-q", "https://github.com/Hamjoon/test-survival-lecture.git", str(LECTURE)], check=True)
if "LECTURE_REPO" not in os.environ:
    for args in (["fetch", "-q"], ["checkout", "-q", BRANCH], ["pull", "-q"]):
        subprocess.run(["git", "-C", str(LECTURE)] + args, check=True)
sys.path.insert(0, str(LECTURE / "tools"))
import lecture as L

L.install_java()

### 1-2. 테스트 실행용 jar 받기

필요한 jar를 작은 `pom.xml`에 적고 Maven으로 받습니다. 테스트 메서드를 이름으로 골라 실행하는 작은 Java 프로그램(`tools/MethodRunner.java`)도 컴파일합니다.

In [ ]:
L.setup_tools()

### 1-3. 실행 도구 점검

실험에서 테스트를 다루는 순서를 작은 예제로 확인합니다: 컴파일 → 실행(JaCoCo로 실행된 줄 기록) → 줄 단위 coverage.

예제 `PriceCalculator`는 할인 서비스에 의존하고, 테스트 두 개 중 `wrongExpectation`은 일부러 기대값을 틀리게 썼습니다. 음수 가격 예외를 던지는 7번째 줄은 어느 테스트도 지나가지 않습니다.

In [ ]:
L.demo()

### 1-4. Gemini API 키 설정

키는 [Google AI Studio](https://aistudio.google.com/apikey)에서 발급합니다. 카드 등록 없이 무료 등급으로 쓸 수 있고, 결제를 따로 켜지 않으면 한도를 넘어도 요금이 나가지 않습니다(요청만 거절됨).

Colab 왼쪽 열쇠 모양(Secrets)에 `GEMINI_API_KEY`라는 이름으로 키를 저장하고 이 노트북의 접근을 허용하세요. 저장하지 않았으면 입력 칸이 뜹니다.

모델이 혼잡하다는 응답(503)이나 한도 초과(429)가 오면 30초, 60초, ... 기다렸다가 자동으로 다시 시도합니다.

모델은 실제 실험 모델(gpt-oss-120b)과 비슷한 등급의 가벼운 모델인 `gemini-3.5-flash-lite`를 씁니다.

In [ ]:
subprocess.run("pip install -q google-genai", shell=True)
from google import genai
import google.genai.errors
from getpass import getpass

API_KEY = None
if L.IN_COLAB:
    try:
        from google.colab import userdata
        API_KEY = userdata.get("GEMINI_API_KEY")
    except Exception:
        API_KEY = None
API_KEY = API_KEY or os.environ.get("GEMINI_API_KEY") or getpass("Gemini API key: ")

MODEL = "gemini-3.5-flash-lite"
client = genai.Client(api_key=API_KEY)
try:
    print("모델:", MODEL, "/ 응답:", L.call_llm(client, MODEL, "Reply with the single word OK.").strip())
except genai.errors.APIError as e:
    print("호출 실패:", e)
    print("\n모델이 계속 혼잡하거나 쓸 수 없으면 MODEL을 아래 중 하나(예: gemini-3.1-flash-lite)로 바꿔 다시 실행하세요.")
    for m in client.models.list():
        if "flash" in m.name and "generateContent" in (m.supported_actions or []):
            print(" ", m.name.removeprefix("models/"))

## 2. 대상 프로젝트와 Dev 테스트

### 2-1. json-schema-validator 받기와 시점 t 빌드 (1~2분)

시점 t는 release 1.1.0(2023-12-15)입니다. 이 시점의 코드로 LLM 테스트를 만들고, 이 시점의 Dev 테스트를 비교 기준으로 씁니다.

In [ ]:
L.checkout_t()
T_CP = L.build_production(L.JSV)
print("의존 jar", len(T_CP), "개")

### 2-2. 대상 클래스

클래스 두 개만 다룹니다. 다른 프로젝트 클래스를 적게 써서 클래스 소스만 보고도 테스트를 쓸 수 있고, 이후 commit에서 코드가 바뀐 클래스입니다.

| 클래스 | 하는 일 |
|---|---|
| `ValidationMessage` | 검증에 실패했을 때 만들어지는 오류 메시지 객체(오류 코드, 위치, 메시지 문자열 등). 객체를 만드는 `Builder`가 안에 있음 |
| `SpecVersionDetector` | 스키마의 `$schema` 값을 보고 어떤 JSON Schema 버전(draft 4, 6, 7, 2019-09, 2020-12)인지 판별 |

전체 소스: [ValidationMessage](https://github.com/networknt/json-schema-validator/blob/1.1.0/src/main/java/com/networknt/schema/ValidationMessage.java), [SpecVersionDetector](https://github.com/networknt/json-schema-validator/blob/1.1.0/src/main/java/com/networknt/schema/SpecVersionDetector.java)

In [ ]:
for c in L.TARGETS:
    L.outline(c)

### 2-3. 비교할 Dev 테스트: Dev(scope)

이 프로젝트의 개발자 테스트는 클래스별로 짜여 있지 않습니다(이슈 번호별 테스트, JSON Schema 공식 테스트 모음 등). 그래서 실제 실험과 같은 방법으로 골랐습니다. t의 Dev 테스트 메서드 207개를 하나씩 단독 실행하고, JaCoCo로 기록해 **통과하면서 대상 클래스 줄을 한 줄이라도 실행한 메서드**만 남겼습니다. 이 집합을 Dev(scope)라고 부릅니다. 시간이 걸려서 목록은 미리 만들어 두었습니다(`prep/dev_scope.py`).

Dev(scope)에는 대상 클래스를 직접 겨냥하지 않는 테스트가 많이 들어 있습니다. 두 클래스는 검증 과정 어디서나 쓰이기 때문에, 예를 들어 공식 테스트 모음은 스키마 전체를 검사하다가 ValidationMessage 줄도 지나갑니다. 반면 LLM 테스트는 대상 클래스를 직접 겨냥해 만듭니다.

In [ ]:
scope = L.load_dev_scope()
DEV_SCOPE = [m["method"] for m in scope["methods"]]
print(f"t의 Dev 테스트 메서드 {scope['dev_methods_total']}개, 단독 실행 통과 {scope['dev_methods_pass']}개, Dev(scope) {len(DEV_SCOPE)}개")
for c in L.TARGETS:
    print(f"  {c} 줄을 실행한 메서드: {sum(c in m['target_lines'] for m in scope['methods'])}개")

### 2-4. 시점 t에서 Dev(scope) 실행

Dev 테스트 소스를 컴파일하고 Dev(scope)를 실행하면서 두 클래스의 줄 coverage를 잽니다. **실행 가능한 줄**은 컴파일된 코드에 대응하는 줄(주석·빈 줄 제외)이고, **coverage**는 그중 테스트가 한 번이라도 지나간 줄의 비율입니다. 뒤에서 LLM 테스트와 비교하는 기준이 됩니다.

In [ ]:
T = L.WORK / "t"
dropped = L.compile_dev_tests(L.JSV / "target/classes", T_CP, T / "dev-test-classes")
exec_t = T / "dev.exec"
exec_t.unlink(missing_ok=True)
dev_t = L.run_methods([T / "dev-test-classes", L.JSV / "target/classes"] + T_CP, DEV_SCOPE, jacoco_exec=exec_t)

print("컴파일 실패한 Dev 테스트 파일:", len(dropped))
print("Dev(scope) 실행 결과:", {s: list(dev_t.values()).count(s) for s in sorted(set(dev_t.values()))})
print("\n시점 t의 Dev(scope) 줄 coverage")
L.print_coverage(L.target_coverage(exec_t, L.JSV / "target/classes", L.JSV / "src/main/java"))

## 3. Ouédraogo 방식

Ouédraogo et al.(EMSE 2026)은 프롬프트 5종(ZSL, FSL, CoT, ToT, GToT)으로 테스트를 만듭니다. 모델에 넘기는 정보는 **대상 클래스의 소스 전체** 하나이고, 프롬프트마다 지시 방식(예시 제공, 단계 지시 등)만 다릅니다. 프롬프트 전문은 [docs/ouedraogo-prompts.md](https://github.com/Hamjoon/test-survival-lecture/blob/main/docs/ouedraogo-prompts.md)에 있습니다.

### 3-1. 생성

두 클래스 × 프롬프트 5종 = 호출 10회입니다(1~2분). temperature는 논문과 같은 0.7입니다. 중간에 끊기면 다시 실행하세요. 이미 받은 응답은 건너뜁니다.

In [ ]:
USE_SAVED = False   # True면 모델을 부르지 않고 강의 저장소에 저장된 응답을 씁니다
L.generate_ouedraogo(client, MODEL, use_saved=USE_SAVED)

### 3-2. 시점 t에서 확인

응답마다 다음을 합니다. 실제 실험과 같은 순서입니다.

1. `###Test START##`와 `###Test END##` 사이의 코드를 꺼내 테스트 파일로 저장합니다.
2. 파일 하나씩 t의 코드에 대해 컴파일합니다. 컴파일에 실패하면 그 파일의 테스트 메서드는 전부 잃습니다.
3. 컴파일된 파일의 테스트 메서드를 t에서 한 번 실행하고, 통과한 것만 남깁니다.
4. 본문이 완전히 같은 메서드는 하나만 남깁니다.

모델이 프롬프트가 지시한 마커를 다르게 쓰면(예: `### Test START ##`처럼 띄어 쓰기) 코드를 꺼내지 못해 그 응답은 추출 실패가 됩니다.

남은 메서드가 이후 commit에서 살아남는지 볼 Ouédraogo 집단입니다.

In [ ]:
rows, OUEDRAOGO = L.check_generated_at_t("ouedraogo", T_CP)
L.show_table(rows)
print("Ouédraogo 집단:", len(OUEDRAOGO), "개 메서드")
# L.save_responses("ouedraogo")   # 응답 원문을 zip으로 내려받기

## 4. KTester 방식

KTester(Li et al., ICSE 2026)는 **focal method**(테스트할 메서드) 하나마다 테스트 클래스 하나를 만듭니다. Ouédraogo와 달리 클래스 소스 외에 두 가지 지식을 프롬프트에 넣습니다.

- **프로젝트 지식**: 프로젝트 전체를 정적 분석해 만든 인덱스에서 꺼낸 생성자, 호출 예시, 의존 클래스 등
- **테스트 지식**: 세 관점(분기, 기능·경계, 예외)으로 테스트 케이스를 먼저 설계하게 함

생성은 논문의 다섯 단계를 따릅니다: test class framework generation → test case design(관점 3개) → test method transformation → test class integration → test class refinement(규칙 수리, LLM 수리 최대 5회). 단계별 설명과 공개 저장소와 다른 점은 [docs/ktester.md](https://github.com/Hamjoon/test-survival-lecture/blob/main/docs/ktester.md)에 있습니다.

프로젝트 지식은 시간이 걸려서 미리 뽑아 두었습니다(`data/ktester/tasks.json`).

### 4-1. 생성

focal method마다 호출이 5~10회입니다. 이미 끝난 focal method는 건너뜁니다.

In [ ]:
import ktester as K
K.setup()
tasks = K.load_tasks()
print("focal method", len(tasks), "개:", ", ".join(t["id"] for t in tasks))

USE_SAVED = False   # True면 모델을 부르지 않고 강의 저장소에 저장된 결과를 씁니다
summary = K.generate_ktester(client, MODEL, T_CP, use_saved=USE_SAVED)
L.show_table(summary)

### 4-2. 시점 t에서 확인

Ouédraogo와 같은 기준으로 확인합니다(파일별 컴파일, t에서 한 번 실행해 통과한 메서드만, 본문이 같은 메서드는 하나만).

In [ ]:
rows, KTESTER = L.check_generated_at_t("ktester", T_CP)
L.show_table(rows)
print("KTester 집단:", len(KTESTER), "개 메서드")
# L.save_responses("ktester"); L.save_responses("ktester-log")   # 결과와 단계별 프롬프트·응답을 zip으로 내려받기